# Entrenamiento del clasificador de estados de tráfico

Usá esta notebook para crear el primer modelo piloto o para reentrenarlo con correcciones humanas. Los dos caminos terminan en el mismo MLP de tres estados y en un bundle portable de cuatro archivos.

| Si estás en esta situación | Elegí | Entrada principal | Resultado |
|---|---|---|---|
| Todavía sólo tenés telemetría cruda | `SEED_BOOTSTRAP` | Backup, CSV o PostgreSQL raw | Modelo piloto + snapshot semilla |
| Ya revisaste inferencias | `HITL_RETRAINING` | Catálogo HITL o PostgreSQL read-only | Candidato entrenado con etiquetas humanas |

**Inicio rápido recomendado:** para el backup histórico dejá la configuración predeterminada, ejecutá `Run All` y cargá `traffic_data.backup` cuando aparezca el selector.

```text
Datos → preparación → etiquetas → particiones
      → balanceo → entrenamiento → evaluación
      → decisión de promoción → bundle
```

<details>
<summary><strong>Ver las seis recetas completas</strong></summary>

### A. Primer modelo desde backup o CSV
```python
TRAINING_MODE = TrainingMode.SEED_BOOTSTRAP
ENABLE_POSTGRES_INGESTION = False
ENABLE_DATA_UPLOAD = True
HUMAN_HOLDOUT_FROZEN = False
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```
Crea o reutiliza la semilla inmutable. El bundle resultante es piloto, no producción.

### B. Primer modelo desde PostgreSQL raw
```python
TRAINING_MODE = TrainingMode.SEED_BOOTSTRAP
ENABLE_POSTGRES_INGESTION = True
ENABLE_DATA_UPLOAD = False
HUMAN_HOLDOUT_FROZEN = False
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```
Necesita el perfil read-only `training`. No escribe datos operacionales.

### C. Reentrenamiento desde el catálogo HITL de Drive
```python
TRAINING_MODE = TrainingMode.HITL_RETRAINING
ENABLE_POSTGRES_INGESTION = False
ENABLE_DATA_UPLOAD = False
HUMAN_HOLDOUT_FROZEN = False
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```
Consume paquetes creados al finalizar revisiones en inferencia. Sólo las filas validadas son targets.

### D. Reentrenamiento combinando catálogo y PostgreSQL
```python
TRAINING_MODE = TrainingMode.HITL_RETRAINING
ENABLE_POSTGRES_INGESTION = True
ENABLE_DATA_UPLOAD = False
HUMAN_HOLDOUT_FROZEN = False
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```
Combina el catálogo con `effective_human_labels` del servidor read-only.

### E. Reutilizar el examen humano congelado
```python
TRAINING_MODE = TrainingMode.HITL_RETRAINING
ENABLE_POSTGRES_INGESTION = False
ENABLE_DATA_UPLOAD = False
HUMAN_HOLDOUT_FROZEN = True
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```
Usa siempre las mismas filas de validation/test para comparar modelos de forma justa.

### F. Crear una nueva generación de holdout o semilla
Cambiá **una sola acción por vez** y escribí un motivo concreto. Nunca se sobrescribe la generación anterior.

**F1. Nuevo holdout humano:**
```python
TRAINING_MODE = TrainingMode.HITL_RETRAINING
ENABLE_POSTGRES_INGESTION = False
ENABLE_DATA_UPLOAD = False
HUMAN_HOLDOUT_FROZEN = True
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.CREATE_NEW_VERSION
HUMAN_HOLDOUT_UPDATE_REASON = "Se incorporaron nuevos episodios revisados"
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
SEED_ARTIFACT_UPDATE_REASON = None
```

**F2. Nueva semilla intencional:**
```python
TRAINING_MODE = TrainingMode.SEED_BOOTSTRAP
ENABLE_POSTGRES_INGESTION = False
ENABLE_DATA_UPLOAD = True
HUMAN_HOLDOUT_FROZEN = False
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
HUMAN_HOLDOUT_UPDATE_REASON = None
SEED_ARTIFACT_ACTION = DatasetArtifactAction.CREATE_NEW_VERSION
SEED_ARTIFACT_UPDATE_REASON = "Se reemplazó la fuente raw inicial"
```

</details>

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.util
import os
import runpy
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORKSPACE_DIR = next(
        (
            path
            for path in candidates
            if (path / "vaaet-core/pyproject.toml").is_file()
            and (path / "vaaet-ml/pyproject.toml").is_file()
        ),
        None,
    )
    if WORKSPACE_DIR is None:
        raise RuntimeError("No se encontró el workspace VAAET con core y ML.")
CORE_ROOT = WORKSPACE_DIR / "vaaet-core"
ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
REPO_ROOT = ML_ROOT
os.chdir(ML_ROOT)
BOOTSTRAP = runpy.run_path(str(ML_ROOT / "scripts" / "notebook_bootstrap.py"))
RUNTIME = BOOTSTRAP["bootstrap_notebook"](
    workspace_root=WORKSPACE_DIR,
    core_root=CORE_ROOT,
    ml_root=ML_ROOT,
    core_extras=('inference',),
    ml_extras=('training', 'visualization', 'database'),
    in_colab=IN_COLAB,
    framework='tensorflow',
    require_gpu=True,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
VAAET_ML_PACKAGE_FILE = RUNTIME.ml_package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Workflow imports — do not edit
import os
import shutil
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from vaaet.artifacts import FEATURE_SCHEMA_VERSION, MANIFEST_FILE, create_manifest
from vaaet_ml.data.database import DatabaseProfile, get_engine, get_optional_database_settings
from vaaet_ml.data.postgres_restore import resolve_pg_restore_for_backup
from vaaet_ml.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet_ml.data.dataset_artifacts import CatalogSelection, DatasetArtifactAction, HitlCatalogSource, SeedArtifactConfig, VersionedSeedStore, create_training_input_lock
from vaaet_ml.data.ingestion import FeedbackPolicy, PostgresBackupSource, PostgresSource, RawCsvSource, SeedDatasetPackageSource, TrainingIngestionPlan, compose_supervised_dataset, load_training_inputs
from vaaet_ml.data.datasets import build_group_ids
from vaaet.timestamps import normalize_timestamp_series
from vaaet.calibration import apply_temperature_scaling, fit_temperature, multiclass_brier_score
from vaaet_ml.evaluation.dataset_validation import audit_training_dataset
from vaaet_ml.evaluation.reporting import build_class_support_notes, build_classification_support_table, expected_calibration_error, expected_confusion_cost, plot_training_evaluation, plot_training_history, select_validation_decision_policy, summarize_data_origin, summarize_state_balance
from vaaet.features.engineering import engineer_features
from vaaet.features.labeling import assign_stable_traffic_state
from vaaet_ml.features.synthetic import augment_with_synthetic
from vaaet.inference.traffic_state import apply_conservative_accident_gate, classify_telemetry_dataframe
from vaaet.logging import configure_logging
from vaaet_ml.workflow_config import TrainingWorkflowConfig
from vaaet_ml.settings import DATA_PROCESSED_DIR, DATA_RAW_DIR, DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABELING_THRESHOLDS, MODEL_DIR, MODEL_VERSION, N_MODEL_STATES, RANDOM_SEED, STATE_LABELS
from vaaet_ml.training.balancing import BalanceStrategy, build_balance_candidates, compute_capped_balanced_weights
from vaaet_ml.training.holdout import HumanHoldoutAction, HumanHoldoutConfig, resolve_human_holdout
from vaaet_ml.training.lifecycle import ModelInputPolicy, TrainingMode, apply_model_input_policy, build_supervision_weights, build_training_lifecycle, cap_synthetic_congested_weight
from vaaet_ml.training.modeling import build_traffic_state_mlp
from vaaet_ml.training.partitions import build_training_partitions
from vaaet_ml.training.selection import select_balance_candidate

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


In [ ]:
# Training workflow configuration — edit only this cell
TRAINING_MODE = TrainingMode.SEED_BOOTSTRAP
# TRAINING_MODE = TrainingMode.HITL_RETRAINING
ENABLE_POSTGRES_INGESTION = False  # Read-only; may be combined with local sources.
ENABLE_DATA_UPLOAD = True  # Set False for a PostgreSQL-only run.
HUMAN_HOLDOUT_FROZEN = False  # Enable only with HITL_RETRAINING and sufficient human support.
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
# HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.CREATE_NEW_VERSION
HUMAN_HOLDOUT_UPDATE_REASON = None  # Required only for CREATE_NEW_VERSION.
SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
# SEED_ARTIFACT_ACTION = DatasetArtifactAction.CREATE_NEW_VERSION
SEED_ARTIFACT_UPDATE_REASON = None  # Required only for an intentional seed change.

if HUMAN_HOLDOUT_FROZEN and TRAINING_MODE is not TrainingMode.HITL_RETRAINING:
    raise ValueError("HUMAN_HOLDOUT_FROZEN=True sólo es válido en HITL_RETRAINING.")

if HUMAN_HOLDOUT_ACTION is HumanHoldoutAction.CREATE_NEW_VERSION and not HUMAN_HOLDOUT_UPDATE_REASON:
    raise ValueError("Para crear un holdout nuevo, explicá el motivo en HUMAN_HOLDOUT_UPDATE_REASON.")
if SEED_ARTIFACT_ACTION is DatasetArtifactAction.CREATE_NEW_VERSION and not SEED_ARTIFACT_UPDATE_REASON:
    raise ValueError("Para crear una semilla nueva, explicá el motivo en SEED_ARTIFACT_UPDATE_REASON.")

_input_summary = (
    'archivo raw subido' if ENABLE_DATA_UPLOAD and TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else 'catálogo HITL de Drive' if TRAINING_MODE is TrainingMode.HITL_RETRAINING
    else 'fuente declarada'
)
if ENABLE_POSTGRES_INGESTION:
    _input_summary += ' + PostgreSQL read-only'
print("✅ Configuración de entrenamiento validada")
print("🧭 Flujo seleccionado")
print(f"   Modo: {TRAINING_MODE.value}")
print(f"   Entrada esperada: {_input_summary}")
print(f"   Holdout humano: {'congelado' if HUMAN_HOLDOUT_FROZEN else 'provisional'}")
print(f"   Acción de semilla: {SEED_ARTIFACT_ACTION.value}")
print("   PostgreSQL: sólo lectura" if ENABLE_POSTGRES_INGESTION else "   PostgreSQL: desactivado")
print("   Salida: métricas + input lock + bundle de cuatro archivos")
print("➡️ Siguiente paso: prepará las fuentes de datos.")
WORKFLOW_CONFIG = TrainingWorkflowConfig(training_mode=TRAINING_MODE.value, enable_postgres_ingestion=ENABLE_POSTGRES_INGESTION, enable_data_upload=ENABLE_DATA_UPLOAD, human_holdout_frozen=HUMAN_HOLDOUT_FROZEN, human_holdout_action=HUMAN_HOLDOUT_ACTION.value, human_holdout_update_reason=HUMAN_HOLDOUT_UPDATE_REASON, seed_artifact_action=SEED_ARTIFACT_ACTION.value, seed_artifact_update_reason=SEED_ARTIFACT_UPDATE_REASON)


## 1. Entender los dos modos

- **Inicio Semilla (`SEED_BOOTSTRAP`)**: toma datos crudos, calcula las 19 features y crea etiquetas provisionales mediante reglas. Es el punto de partida rápido.
- **Reentrenamiento HITL (`HITL_RETRAINING`)**: toma features ya calculadas y correcciones humanas. No vuelve a hacer ingeniería sobre esos registros.

<details>
<summary><strong>Diccionario en palabras simples</strong></summary>

- **Weak supervision:** reglas que crean etiquetas provisionales cuando todavía no hay suficientes revisiones humanas.
- **Memoria proxy decreciente:** al principio conserva parte de la semilla; a medida que llegan etiquetas humanas, esa influencia baja por clase hasta desaparecer.
- **Datos sintéticos:** ejemplos artificiales usados sólo en train para ejercitar congestión e incidentes; nunca prueban calidad real.
- **Holdout humano:** un examen fijo que el modelo nunca usa para aprender. Permite comparar candidatos con las mismas preguntas.
- **Snapshot semilla:** fotografía procesada e inmutable de los datos iniciales.
- **Catálogo HITL:** lista verificada de paquetes producidos por sesiones de revisión.
- **Training input lock:** comprobante exacto de qué semilla, feedback y holdout usó un entrenamiento.
- **Pilot / candidate / production:** piloto sirve para iniciar el ciclo; candidato espera evaluación; producción superó todos los gates.

</details>

Las fuentes son explícitas: backup/CSV/PostgreSQL para raw; snapshot semilla y catálogo/PostgreSQL para HITL. Nunca se adivina el tipo por sus columnas y una predicción sin revisar jamás se convierte en etiqueta.

PostgreSQL es siempre read-only y sólo se consulta con `ENABLE_POSTGRES_INGESTION=True`. Configurá el perfil `training` según la [guía canónica de Colab](../../../docs/operations/colab-guide.md#secrets-y-postgresql).

## 2. Preparar archivos y fuentes

En modo semilla, Drive guarda el snapshot inmutable y Colab solicita el backup o CSV sólo cuando hace falta. En modo HITL, se reutilizan la semilla y el catálogo de revisiones de Drive. La siguiente etapa muestra exactamente qué fuente encontró.

In [ ]:
# Cell 1b — Data Upload (Colab only)
#
# On Colab, if no CSV cache exists, upload one of:
#   - traffic_data.backup  (pg_dump binary → processed via pg_restore)
#   - traffic_data_raw.csv (explicit RawCsvSource)
# HITL retraining reads the immutable Drive catalog; it does not upload a mutable ZIP.
# The processed seed is resolved through VersionedSeedStore/current.json.
# On local, configure the equivalent filesystem-backed artifact roots below.
_backup_dest = os.path.join(_RAW_DIR, "traffic_data.backup")
_csv_dest = os.path.join(_RAW_DIR, "traffic_data_raw.csv")
HUMAN_HOLDOUT_STORE_ROOT = _DATA_DIR / "holdouts"
SEED_ARTIFACT_ROOT = _DATA_DIR / "seed-bootstrap"
HITL_CATALOG_PATH = _DATA_DIR / "hitl-reviews/catalog.json"
TRAINING_RUNS_ROOT = _DATA_DIR / "training-runs"
if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError("Los datasets inmutables necesitan Google Drive montado; no se usará un fallback efímero.") from exc
    _drive_data_root = Path("/content/drive/MyDrive/vaaet-ml/data")
    SEED_ARTIFACT_ROOT = _drive_data_root / "seed-bootstrap"
    HITL_CATALOG_PATH = _drive_data_root / "hitl-reviews/catalog.json"
    TRAINING_RUNS_ROOT = Path("/content/drive/MyDrive/vaaet-ml/training-runs")
    HUMAN_HOLDOUT_STORE_ROOT = Path("/content/drive/MyDrive/vaaet-ml/data/holdouts")
    for _artifact_directory in (SEED_ARTIFACT_ROOT, HITL_CATALOG_PATH.parent, TRAINING_RUNS_ROOT, HUMAN_HOLDOUT_STORE_ROOT):
        _artifact_directory.mkdir(parents=True, exist_ok=True)
    print(f"🔒 Datos inmutables en Drive: {_drive_data_root}")

_required_local_input_exists = (
    any(os.path.exists(path) for path in (_csv_dest, _backup_dest))
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else HITL_CATALOG_PATH.is_file()
)


In [ ]:
if IN_COLAB and ENABLE_DATA_UPLOAD and not _required_local_input_exists:
    from google.colab import files  # type: ignore[import-untyped]
    if TRAINING_MODE is TrainingMode.HITL_RETRAINING:
        raise FileNotFoundError(f"No existe un catálogo HITL en {HITL_CATALOG_PATH}. Primero finalizá revisiones en la notebook de inferencia.")
    print("📤 Subí un backup PostgreSQL raw o un CSV de telemetría raw:")
    uploaded = files.upload()
    if uploaded:
        import shutil as _shutil
        for fname in uploaded:
            if fname.endswith(".csv"):
                _shutil.move(fname, _csv_dest)
                print(f"✅ CSV guardado en {_csv_dest}; se declarará como RawCsvSource")
            else:
                _shutil.move(fname, _backup_dest)
                print(f"✅ Backup guardado en {_backup_dest}; se declarará como PostgresBackupSource")
    else:
        print("⚠️ No se subió ningún archivo. Usá PostgreSQL o repetí esta celda.")
else:
    if os.path.exists(_csv_dest):
        print(f"📂 CSV disponible: {os.path.abspath(_csv_dest)}")
    elif os.path.exists(_backup_dest):
        print(f"📂 Backup disponible: {os.path.abspath(_backup_dest)}")
    elif HITL_CATALOG_PATH.is_file():
        print(f"📚 Catálogo HITL disponible: {HITL_CATALOG_PATH}")
    elif not ENABLE_DATA_UPLOAD:
        print("ℹ️ Upload desactivado; la siguiente etapa buscará PostgreSQL.")
    else:
        print("📂 No hay una fuente local disponible.")

# A binary pg_dump needs a compatible PostgreSQL client (OS dependency).
PG_RESTORE_PATH: str | None = shutil.which("pg_restore")


In [ ]:
# Cell 1c — Resolve the explicit PostgreSQL backup reader
PG_RESTORE_PATH = resolve_pg_restore_for_backup(
    Path(_backup_dest),
    Path(_csv_dest),
    in_colab=IN_COLAB,
)


In [ ]:
print("➡️ Siguiente paso: cargá y validá las fuentes declaradas.")

In [ ]:
# Cell 2A/2B — Explicit seed or HITL ingestion
# Change only TRAINING_MODE and the corresponding typed source list.

RAW_CSV_PATH = _RAW_DIR / "traffic_data_raw.csv"
BACKUP_PATH = _RAW_DIR / "traffic_data.backup"
SEED_STORE = VersionedSeedStore(SEED_ARTIFACT_ROOT)
seed_snapshot = SEED_STORE.load_current()
HITL_CATALOG_DESCRIPTOR = None
training_db_settings = None
if ENABLE_POSTGRES_INGESTION:
    training_db_settings = get_optional_database_settings(DatabaseProfile.TRAINING)
    if training_db_settings is None:
        raise RuntimeError(
            "PostgreSQL ingestion is enabled, but the read-only training profile is not configured. "
            "Configure the read-only training profile according to the canonical Colab guide."
        )
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    RAW_SOURCES = []
    if BACKUP_PATH.is_file():
        RAW_SOURCES.append(
            PostgresBackupSource(BACKUP_PATH, Path(PG_RESTORE_PATH) if PG_RESTORE_PATH else None)
        )
    if RAW_CSV_PATH.is_file():
        RAW_SOURCES.append(RawCsvSource(RAW_CSV_PATH))
    if training_db_settings is not None:
        RAW_SOURCES.append(PostgresSource(training_db_settings))
    if not RAW_SOURCES:
        raise RuntimeError(
            "No seed source is available. Upload a raw backup/CSV or enable PostgreSQL ingestion."
        )
    SEED_SOURCES = []
    FEEDBACK_SOURCES = []
else:
    RAW_SOURCES = []  # standard HITL flow reuses processed features
    if seed_snapshot is None:
        raise FileNotFoundError(f"No existe una semilla inmutable activa en {SEED_ARTIFACT_ROOT}. Primero ejecutá SEED_BOOTSTRAP.")
    SEED_SOURCES = [SeedDatasetPackageSource(seed_snapshot.path)]
    FEEDBACK_SOURCES = []
    if HITL_CATALOG_PATH.is_file():
        FEEDBACK_SOURCES.append(
            HitlCatalogSource(HITL_CATALOG_PATH, CatalogSelection.ALL_ACTIVE)
        )
    if training_db_settings is not None:
        FEEDBACK_SOURCES.append(PostgresSource(training_db_settings))
    if not FEEDBACK_SOURCES:
        raise RuntimeError(
            "No HITL feedback source is available. Finalize review packages or enable PostgreSQL ingestion."
        )


In [ ]:

TRAINING_INPUTS = TrainingIngestionPlan(
    mode=TRAINING_MODE,
    raw_sources=tuple(RAW_SOURCES),
    seed_sources=tuple(SEED_SOURCES),
    feedback_sources=tuple(FEEDBACK_SOURCES),
    feedback_policy=FeedbackPolicy.VALIDATED_ONLY,
)
training_inputs = load_training_inputs(TRAINING_INPUTS)
df_raw = training_inputs.raw
seed_feature_frame = training_inputs.seed_features
validated_feedback = training_inputs.validated_feedback
confirmed_incidents = training_inputs.confirmed_incidents
DATA_SOURCE = ','.join(training_inputs.provenance['source_type'].astype(str))
display(training_inputs.provenance)
_catalog_rows = training_inputs.provenance.loc[training_inputs.provenance['source_type'].eq('HitlCatalogSource')]
if not _catalog_rows.empty:
    _catalog_record = _catalog_rows.iloc[-1].to_dict()
    HITL_CATALOG_DESCRIPTOR = {key: _catalog_record[key] for key in ('contract', 'revision', 'catalog_sha256', 'package_ids', 'package_fingerprints', 'package_sha256', 'resolved_validations', 'duplicate_rows_resolved', 'corrections_resolved') if key in _catalog_record}
for _source in training_inputs.provenance.to_dict(orient='records'):
    if _source.get('archive_table'):
        print(
            f"Detected backup table: {_source['archive_table']} "
            f"({_source['backup_layout']} raw telemetry) | "
            f"reader={_source['reader_version']} | imported rows={_source['rows']}"
        )
print(f"Modo: {TRAINING_MODE.value}")
print(f"Filas raw: {len(df_raw)} | semilla procesada: {len(seed_feature_frame)} | feedback estable validado: {len(validated_feedback)} | incidentes confirmados: {len(confirmed_incidents)}")
if not df_raw.empty:
    print(f"Rango temporal raw: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")
print("➡️ Siguiente paso: prepará la augmentación exclusiva del inicio semilla.")


In [ ]:
# Cell 2b — Synthetic Data Augmentation
#
# The Belgrano Bridge dataset (Apr–Jul 2025) has no human-confirmed
# Accident and only limited proxy support for Congested.
# We inject physically plausible synthetic sequences so the classifier
# can stress Congested and possible-incident boundaries. Synthetic IDs start at 50001.
# fall before the real data range (2025-04-21…27).

if "df_raw" not in globals() or not isinstance(df_raw, pd.DataFrame):
    raise RuntimeError("Primero ejecutá la carga de datos.")
_n_before = len(df_raw)
if TRAINING_MODE is TrainingMode.HITL_RETRAINING or df_raw.empty:
    _n_synthetic = 0
    print("ℹ️ Datos sintéticos omitidos: sólo se agregan durante el inicio semilla.")
else:
    df_raw = augment_with_synthetic(
        df_raw, n_accident_seq=10, n_congestion_seq=10, records_per_seq=10, seed=RANDOM_SEED
    )
    _n_synthetic = len(df_raw) - _n_before
    print(f"   Zona horaria canónica: {df_raw['record_time'].dt.tz}")

print(f"✅ Augmentación sintética: {_n_synthetic} registros agregados")
print("   Accident: 10 secuencias × 10 = 100 registros de stress técnico")
print("   Congested: 10 secuencias × 10 = 100 registros de entrenamiento")
print(f"   Dataset total: {len(df_raw)} registros ({_n_before} reales + {_n_synthetic} sintéticos)")

if not df_raw.empty:
    origin_summary = summarize_data_origin(df_raw)
    print("\n📋 Procedencia del dataset:")
    display(origin_summary) if "display" in dir() else print(origin_summary.to_string(index=False))
print("➡️ Siguiente paso: calculá o validá las 19 features.")


## 3. Convertir telemetría cruda en 19 features

Los datos raw traen velocidades y conteos. La ingeniería agrega relaciones entre minutos para que el modelo pueda reconocer cambios, acumulación y persistencia.

| Feature | Origen | Qué aporta |
|---|---|---|
| `avg_speed` | Directo | Velocidad media del flujo |
| `total_vehicles` | Directo | Volumen total por minuto |
| Conteos por tipo (5) | Directo | Composición del tránsito |
| `heavy_vehicle_ratio` | Derivado | Proporción de vehículos pesados |
| `delta_speed`, `delta_count` | Derivado | Cambio frente al minuto anterior |
| `transition_flag` | Derivado | Cambio brusco simultáneo de velocidad y volumen |
| `speed_variance` | Derivado | Estabilidad reciente de la velocidad |
| `cumulative_delta_speed` | Derivado | Tendencia acumulada dentro del clip |
| `low_speed_persistence` | Derivado | Cuánto dura una condición lenta |
| Señales de calidad y movimiento (3) | Directo/derivado | Confiabilidad y vehículos detenidos |
| `hour_of_day`, `weather_condition` | Temporal | Contexto horario y ambiental |

El primer registro de cada secuencia no tiene un minuto anterior para calcular diferencias, por eso se descarta de forma controlada.

> **Nota**: el inicio semilla agrega 200 registros sintéticos (100 de congestión y 100 de stress de incidente). Feature engineering los procesa con la misma semántica temporal. Accident se separa antes del target; Congested sintético sólo puede entrar en train con peso reducido.

In [ ]:
# Cell 3 — Feature Engineering
#
# Uses vaaet.features.engineering.engineer_features() and vaaet_ml.settings.FEATURE_COLS
# (imported in Cell 1). No inline duplication.

audit_frame = (
    (df_raw if not df_raw.empty else seed_feature_frame)
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else validated_feedback
)
dataset_audit = audit_training_dataset(audit_frame, require_production_eligible=False)
print("📋 Auditoría previa al entrenamiento:")
print(json.dumps(dataset_audit.report, indent=2, default=str))

engineered_proxy_frame = engineer_features(df_raw) if not df_raw.empty else validated_feedback.head(0).copy()
legacy_missing = [column for column in FEATURE_COLS if not engineered_proxy_frame.empty and engineered_proxy_frame[column].isna().any()]
if legacy_missing:
    print("⚠️ Las filas legacy v1 no contienen evidencia moderna de calidad.")
    print("   Este entrenamiento puede crear un piloto experimental, nunca producción.")
    print(f"   Neutralización conservadora de columnas desconocidas: {legacy_missing}")
    engineered_proxy_frame[legacy_missing] = engineered_proxy_frame[legacy_missing].fillna(0.0)

# Save features CSV for reproducibility (NOT to be used as raw data fallback)
csv_path = os.path.join(_DATA_DIR, "traffic_telemetry.csv")
engineered_proxy_frame.to_csv(csv_path, index=False)

print(f"✅ Features listas: {engineered_proxy_frame.shape[0]} filas × {engineered_proxy_frame.shape[1]} columnas")
print(f"   CSV reproducible → {os.path.abspath(csv_path)}")
print("\n📊 Correlación con avg_speed:")
if not engineered_proxy_frame.empty:
    corr = engineered_proxy_frame[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
    print(corr.to_string())
else:
    print("Modo HITL sin raw: no corresponde calcular correlación sobre filas raw.")
print("➡️ Siguiente paso: asigná o incorporá las etiquetas.")

## 4. Crear etiquetas provisionales

En el inicio semilla todavía no hay miles de etiquetas humanas. Por eso se usa una matriz de reglas como verdad provisional, calibrada con la distribución observada en el Puente Belgrano.

- **Accident (3)** no es una salida del MLP. Los escenarios sintéticos de incidente se reservan para pruebas técnicas del detector jerárquico.
Los valores concretos no se duplican en Markdown: la siguiente celda imprime la matriz vigente directamente desde `LABELING_THRESHOLDS`, única fuente de verdad.

Los datos sintéticos de Congested sólo pueden aumentar `train`; nunca forman parte de validation/test. Estas reglas son etiquetas proxy y no sustituyen ground truth humano.

**Límite importante:** una etiqueta proxy no es verdad humana. Sólo `vaaet_feedback.human_validations` aporta ground truth; una predicción sin revisar nunca es target. Ver [sesgos y limitaciones](../../../docs/ml/bias-and-limitations.md).

In [ ]:
# Cell 4 — Auto-Labeling + Class Distribution
#
# Uses the stable three-class proxy labeler. Accident is never an MLP target.
# (imported in Cell 1). No inline duplication.

print("📐 Matriz activa de etiquetas provisionales:")
print(json.dumps(dict(LABELING_THRESHOLDS), indent=2))
scenario = engineered_proxy_frame.get("synthetic_scenario", pd.Series("observed", index=engineered_proxy_frame.index))
incident_stress_frame = engineered_proxy_frame.loc[scenario.eq("accident")].copy()
new_proxy_features = engineered_proxy_frame.loc[~scenario.eq("accident")].copy()
if not new_proxy_features.empty:
    new_proxy_features["traffic_state"] = assign_stable_traffic_state(new_proxy_features)
    new_proxy_features["is_human_validated"] = False
    new_proxy_features["feature_schema_version"] = FEATURE_SCHEMA_VERSION
proxy_frames = [frame for frame in (seed_feature_frame, new_proxy_features) if not frame.empty]
proxy_features = pd.concat(proxy_frames, ignore_index=True) if proxy_frames else validated_feedback.head(0).copy()
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP and not new_proxy_features.empty:
    metadata_columns = [column for column in new_proxy_features if column not in FEATURE_COLS]
    seed_export = new_proxy_features[[*metadata_columns, *FEATURE_COLS]]
    seed_snapshot = SEED_STORE.resolve(
        seed_export,
        SeedArtifactConfig(
            store_root=SEED_ARTIFACT_ROOT,
            action=SEED_ARTIFACT_ACTION,
            update_reason=SEED_ARTIFACT_UPDATE_REASON,
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
        ),
    )
    print(f"💾 Snapshot semilla inmutable → {seed_snapshot.path}")
    print(json.dumps(seed_snapshot.descriptor, indent=2))
    SEED_ARTIFACT_ACTION = DatasetArtifactAction.REUSE_OR_CREATE
    SEED_ARTIFACT_UPDATE_REASON = None
df_features = compose_supervised_dataset(proxy_features, validated_feedback)
if not validated_feedback.empty:
    print(f"✅ Se incorporaron {len(validated_feedback)} etiquetas humanas estables; {len(confirmed_incidents)} incidentes confirmados quedaron fuera del MLP.")

# Distribution
dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Distribución de estados:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} records ({pct:.1f}%)")

# Verify at least 2 classes exist
n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Sólo se encontró una clase: los umbrales no separan este dataset.")
else:
    print(f"\n✅ Se detectaron {n_classes} clases")


In [ ]:

# Classes without samples
for code, label in {code: STATE_LABELS[code] for code in range(N_MODEL_STATES)}.items():
    if code not in dist.index:
        print(f"⚠️ La clase '{label}' ({code}) no tiene ejemplos y se excluirá.")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Registros")
ax.set_title("Distribución de estados (etiquetado provisional)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

support_summary = summarize_state_balance(df_features)
print("\n📋 Soporte por procedencia:")
display(support_summary) if "display" in dir() else print(support_summary.to_string(index=False))

print("\n📝 Notas sobre el soporte:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")
print("➡️ Siguiente paso: verificá el feedback humano disponible.")


## 5. Incorporar correcciones humanas (modo HITL)

Las correcciones humanas entran en este punto y prevalecen sobre etiquetas proxy coincidentes. `Accident` confirmado se reserva para evaluar el detector de incidentes y nunca se convierte en target del MLP.


In [ ]:
# Cell 4b — Explicit HITL ingestion report
print(f"Etiquetas humanas estables incluidas: {len(validated_feedback)}")
print(f"Incidentes confirmados reservados fuera del MLP: {len(confirmed_incidents)}")
if validated_feedback.empty:
    print("ℹ️ No hay feedback humano: los gates de producción permanecerán bloqueados.")
print("➡️ Siguiente paso: construí particiones sin leakage.")


## 6. Separar datos y balancear sin leakage

El clasificador aprende únicamente Normal, Reduced y Congested. Accident se excluye del target y se gestiona con la política jerárquica.

1. **Scaler:** normaliza las 19 features usando únicamente train.
2. **Train/validation/test:** mantiene clips completos y reserva los grupos temporales posteriores para test.
3. **Balanceo conservador:** compara alternativas en validation; los sintéticos sólo aparecen en train.

El scaler se exporta como `feature_scaler.joblib` para que inferencia aplique exactamente la misma transformación.

In [ ]:
# Cell 5 — Temporal Group Split + Conservative Class Weighting

human_holdout_snapshot = None
if HUMAN_HOLDOUT_FROZEN:
    human_holdout_snapshot = resolve_human_holdout(
        validated_feedback,
        HumanHoldoutConfig(
            store_root=HUMAN_HOLDOUT_STORE_ROOT,
            action=HUMAN_HOLDOUT_ACTION,
            update_reason=HUMAN_HOLDOUT_UPDATE_REASON,
            validation_size=0.2,
            test_size=0.2,
            random_state=RANDOM_SEED,
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
        ),
    )
    print(f"🔒 Frozen human holdout: {json.dumps(human_holdout_snapshot.descriptor)}")
    HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
    HUMAN_HOLDOUT_UPDATE_REASON = None

partitions = build_training_partitions(
    proxy_features, validated_feedback, TRAINING_MODE,
    test_size=0.2, validation_size=0.2, random_state=RANDOM_SEED,
    frozen_holdout=human_holdout_snapshot,
)
train_frame = partitions.train
validation_frame = partitions.validation
test_frame = partitions.test

supervision_weight, supervision_report = build_supervision_weights(train_frame, TRAINING_MODE)
active_supervision = supervision_weight > 0
discarded_proxy_rows = int((~active_supervision).sum())
if discarded_proxy_rows:
    train_frame = train_frame.loc[active_supervision].copy()
    supervision_weight = supervision_weight[active_supervision]
    print(f"ℹ️ Removed {discarded_proxy_rows} expired proxy-memory rows before scaling.")
has_effective_proxy_memory = bool(((~train_frame["is_human_validated"].fillna(False)) & pd.Series(supervision_weight > 0, index=train_frame.index)).any())
MODEL_INPUT_POLICY = (
    ModelInputPolicy.LEGACY_V1_BOOTSTRAP
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP or has_effective_proxy_memory
    else ModelInputPolicy.CANONICAL_V2
)
X_train_raw = apply_model_input_policy(train_frame, MODEL_INPUT_POLICY).to_numpy()
X_validation_raw = apply_model_input_policy(validation_frame, MODEL_INPUT_POLICY).to_numpy()
X_test_raw = apply_model_input_policy(test_frame, MODEL_INPUT_POLICY).to_numpy()
y_train = train_frame["traffic_state"].to_numpy(dtype=int)
y_validation = validation_frame["traffic_state"].to_numpy(dtype=int)
y_test = test_frame["traffic_state"].to_numpy(dtype=int)


In [ ]:

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_validation = scaler.transform(X_validation_raw)
X_test = scaler.transform(X_test_raw)

balance_candidates = build_balance_candidates(
    train_frame, supervision_weight, random_state=RANDOM_SEED
)
synthetic_train = train_frame.get("data_origin", pd.Series("real", index=train_frame.index)).eq("synthetic").to_numpy()

print("📊 Partición protegida contra leakage:")
print(f"   Train={len(train_frame)} | Validation={len(validation_frame)} | Test={len(test_frame)}")
print(f"   Alternativas de balanceo: {[strategy.value for strategy in balance_candidates]}")
print(f"   Política de entrada: {MODEL_INPUT_POLICY.value}")
print(f"   Política de supervisión: {json.dumps(supervision_report, default=str)}")
print(f"   Sintéticos en validation/test: {(validation_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in validation_frame else 0}/{(test_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in test_frame else 0}")
partition_summary = pd.DataFrame([
    {"partition": name, "records": len(frame), "clips": build_group_ids(frame).nunique(), "start": normalize_timestamp_series(frame["record_time"]).min(), "end": normalize_timestamp_series(frame["record_time"]).max(), "missing_features": int(frame[FEATURE_COLS].isna().sum().sum())}
    for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
])
display(partition_summary) if "display" in dir() else print(partition_summary.to_string(index=False))

# Export scaler
scaler_path = os.path.join(_MODEL_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler guardado → {os.path.abspath(scaler_path)}")

print("✅ Sin SMOTE por defecto: validation y test permanecen reales e intactos.")
print("➡️ Siguiente paso: entrená y compará las alternativas de balanceo.")


## 7. Entrenar el MLP tabular

El modelo es un **MLP** deliberadamente simple. Aprende combinaciones no lineales de las 19 features sin reemplazar la política temporal del pipeline.

- `Dense(64) → Dense(32)`: aprende y comprime relaciones entre features.
- `BatchNormalization` y `Dropout`: estabilizan y reducen sobreajuste.
- `Dense(3, softmax)`: produce Normal, Reduced o Congested. Nunca Accident.

In [ ]:
# Cell 6 — Model Definition + Training

n_classes: int = N_MODEL_STATES
n_features: int = X_train.shape[1]

print(f"🏗️ Construyendo MLP: {n_features} features → {n_classes} clases")
print("   Salidas: Normal / Reduced / Congested. Accident no se aprende.")

model = build_traffic_state_mlp(input_features=n_features, output_classes=n_classes)

model.summary()

# Train and compare all conservative balance alternatives on validation only.
print("\n🚀 Comparando estrategias conservadoras de balanceo...")
_training_engine = get_engine(training_db_settings) if training_db_settings is not None else None
_training_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.TRAINING, git_commit=GIT_COMMIT, source_kind="composed-dataset", input_rows=len(X_train), model_version=MODEL_VERSION)


In [ ]:
# Cell 6a — Select conservative balance by validation-only evidence
try:
    with pipeline_run(
        _training_metadata,
        engine=_training_engine,
        local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs",
    ) as _run:
        _selection = select_balance_candidate(
            candidates=balance_candidates, train_frame=train_frame, x_train=X_train,
            y_train=y_train, x_validation=X_validation, y_validation=y_validation,
            validation_frame=validation_frame, scaler=scaler, input_policy=MODEL_INPUT_POLICY,
            input_features=n_features, output_classes=n_classes, random_seed=RANDOM_SEED,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-6, verbose=0),
            ],
            clear_session=tf.keras.backend.clear_session, set_random_seed=tf.random.set_seed,
        )
        _run.set_output_rows(len(balance_candidates[_selection.strategy].row_positions))
    TRAINING_PIPELINE_RUN_ID = str(_run.id)
finally:
    if _training_engine is not None:
        _training_engine.dispose()

balance_selection_report = _selection.report
SELECTED_BALANCE_STRATEGY = _selection.strategy
model, history = _selection.model, _selection.history
sample_weight, class_weights = _selection.sample_weight, _selection.class_weights


In [ ]:

display(balance_selection_report) if "display" in dir() else print(balance_selection_report.to_string(index=False))
print(f"✅ Estrategia elegida: {SELECTED_BALANCE_STRATEGY.value}")
model.summary()

plot_training_history(history.history)

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Entrenamiento terminado — mejor época: {best_epoch}")
print("➡️ Siguiente paso: evaluá el candidato sobre test.")

## 8. Evaluar el candidato

La exactitud global puede ocultar fallos en clases pequeñas. Por eso se muestran:

- **F1-macro de tres estados** ≥ 0.88, siempre acompañado por soporte real y número de clips
- **Coste de confusión**: penaliza especialmente los errores directos Normal ↔ Congested
- **ECE**: impide presentar softmax como probabilidad fiable sin comprobar calibración
- **Matriz de confusión**: muestra qué estados se confunden entre sí

Accident no tiene recall publicable sin casos reales. El bundle se marca experimental mientras falte telemetría v2 y holdout humano.

In [ ]:
# Cell 7 — Evaluation + Model Export

# Calibrate and select thresholds on validation only.
validation_proba_raw = model.predict(X_validation, verbose=0)
temperature = fit_temperature(validation_proba_raw, y_validation)
validation_proba = apply_temperature_scaling(validation_proba_raw, temperature)
decision_policy = select_validation_decision_policy(validation_frame, y_validation, validation_proba, temperature=temperature)
print(f"🎛️ Política elegida exclusivamente con validation: {decision_policy}")

# Evaluate the exact production chain on the frozen real test groups.
y_proba_raw = model.predict(X_test, verbose=0)
y_proba = apply_temperature_scaling(y_proba_raw, temperature)
classified_test = classify_telemetry_dataframe(test_frame, model, scaler, decision_policy=decision_policy, input_policy=MODEL_INPUT_POLICY)
y_model_pred = y_proba.argmax(axis=1).astype(int)
y_pred = classified_test["traffic_state"].to_numpy(dtype=int)
direct_target_accuracy = float((y_model_pred == y_test).mean())
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    weak_supervision_fidelity = direct_target_accuracy
    human_holdout_direct_accuracy = None
    print(f"Fidelidad a las reglas provisionales (MLP directo): {direct_target_accuracy:.2%}")
else:
    weak_supervision_fidelity = None
    human_holdout_direct_accuracy = direct_target_accuracy
    print(f"Exactitud directa sobre holdout humano: {direct_target_accuracy:.2%}")

# Present class names
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

# Classification Report
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)
support_with_intervals = build_classification_support_table(y_test, y_pred)
print("\nSoporte por clase e intervalos Wilson del 95%:")
display(support_with_intervals) if "display" in dir() else print(support_with_intervals.to_string(index=False))

# F1-macro
f1_macro = f1_score(y_test, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)
confusion_cost = expected_confusion_cost(y_test, y_pred)
ece = expected_calibration_error(y_test, y_proba)
brier = multiclass_brier_score(y_test, y_proba)
direct_extreme_error = float((((y_test == 0) & (y_pred == 2)) | ((y_test == 2) & (y_pred == 0))).mean())


In [ ]:
automatic_accident_states = int(classified_test["traffic_state"].eq(3).sum())
incident_candidate_count = int(classified_test["accident_alert_started"].sum())
reliable_negative_mask = classified_test["measurement_reliable"].fillna(False).astype(bool)
negative_exposure_hours = float(reliable_negative_mask.sum()) / 60.0
reliable_incident_candidates = int((classified_test["accident_alert_started"] & reliable_negative_mask).sum())
false_candidates_per_hour = reliable_incident_candidates / negative_exposure_hours if negative_exposure_hours else float("nan")
if automatic_accident_states != 0:
    raise RuntimeError("Se violó una garantía: Accident fue producido automáticamente.")
synthetic_incident_episode_sensitivity = None
if not incident_stress_frame.empty:
    stress = incident_stress_frame.copy()
    stress["traffic_state"] = 2
    stress["state_label"] = STATE_LABELS[2]
    stress["confidence"] = 0.5
    stress = apply_conservative_accident_gate(stress)
    episode_hits = stress.groupby("clip_id")["accident_alert_started"].any()
    synthetic_incident_episode_sensitivity = float(episode_hits.mean())
    print(f"Sensibilidad sobre stress sintético de incidentes: {synthetic_incident_episode_sensitivity:.2%} (prueba técnica, no recall real)")
confirmed_incident_candidate_sensitivity = None
confirmed_incident_support = len(confirmed_incidents)
if confirmed_incident_support:
    human_incident_context = pd.concat([validated_feedback, confirmed_incidents], ignore_index=True).sort_values(["clip_id", "record_time"])
    human_incident_context["traffic_state"] = 2
    human_incident_context["state_label"] = STATE_LABELS[2]
    human_incident_context["confidence"] = 0.5
    human_incident_context = apply_conservative_accident_gate(human_incident_context)
    confirmed_keys = set(zip(confirmed_incidents["clip_id"], normalize_timestamp_series(confirmed_incidents["record_time"])))
    evaluated_keys = list(zip(human_incident_context["clip_id"], normalize_timestamp_series(human_incident_context["record_time"])))
    confirmed_mask = pd.Series([key in confirmed_keys for key in evaluated_keys], index=human_incident_context.index)
    confirmed_incident_candidate_sensitivity = float(human_incident_context.loc[confirmed_mask, "accident_rule_triggered"].mean())
    print(f"Detección candidata sobre incidentes confirmados: {confirmed_incident_candidate_sensitivity:.2%} (soporte={confirmed_incident_support}; todavía no es recall operacional)")
else:
    print("Detección de incidentes confirmados: sin soporte (0 casos humanos)")
print(f"{'F1-macro':>15}: {f1_macro:.4f}")
print(f"{'Coste esperado':>15}: {confusion_cost:.4f}")
print(f"{'ECE':>15}: {ece:.4f}")
print(f"{'Brier':>15}: {brier:.4f}")
print(f"{'Normal↔Congested':>15}: {direct_extreme_error:.2%}")
print(f"{'Candidatos/hora':>15}: {false_candidates_per_hour:.6f} sobre {negative_exposure_hours:.2f} h")
if negative_exposure_hours < 300:
    print("⚠️ La tasa de falsas alertas es preliminar; se necesitan unas 300 horas negativas.")

if f1_macro >= 0.88:
    print("✅ F1-macro cumple el objetivo (≥ 0.88)")
else:
    print("⚠️ F1-macro no alcanza 0.88; el bundle continúa experimental.")

print("➡️ Siguiente paso: revisá matrices y confiabilidad.")


### 8.1 Entender dónde se equivoca

Las matrices comparan la salida directa del MLP con el estado final después de umbrales e histéresis. El diagrama de confiabilidad muestra si una confianza alta realmente coincide con una mayor tasa de aciertos.

In [ ]:
# Cell 8b — Visual evaluation
plot_training_evaluation(
    y_test, y_model_pred, y_pred, y_proba, state_labels=STATE_LABELS
)

# Per-class recall
print("\n📊 Recall por estado:")
for row in support_with_intervals.itertuples(index=False):
    status = "✅" if row.recall > 0 else "🔴"
    print(f"   {status} {row.state_label:>10}: {row.recall:.4f} (soporte={row.support})")

print("➡️ Siguiente paso: calculá elegibilidad y exportá el bundle.")


## 9. Decidir promoción y exportar el bundle

Esta etapa no aprueba un modelo por intuición. Reúne blockers, genera el input lock, escribe el manifiesto y deja claro si el resultado es `pilot`, `candidate` o `production`.

In [ ]:
# Cell 9 — Promotion decision and bundle export
# Export model
model_path = os.path.join(_MODEL_DIR, "traffic_classifier.keras")
model.save(model_path)

# Export the canonical mapping required by the portable serving contract.
label_mapping = dict(STATE_LABELS)
label_path = os.path.join(_MODEL_DIR, "label_mapping.joblib")
joblib.dump(label_mapping, label_path)
human_test_only = bool("is_human_validated" in test_frame and test_frame["is_human_validated"].fillna(False).all())
human_holdout = bool(human_test_only and human_holdout_snapshot is not None)
promotion_blockers = list(dataset_audit.blockers)
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    promotion_blockers.append("seed bootstrap uses weak proxy supervision and is pilot-only")
if not human_holdout:
    promotion_blockers.append("validation/test are not a frozen human-validated holdout")
metric_report = classification_report(y_test, y_pred, labels=[0, 1, 2], output_dict=True, zero_division=0)
metric_gates = {
    "f1_macro": f1_macro >= 0.88,
    "normal_precision": metric_report["0"]["precision"] >= 0.93,
    "normal_recall": metric_report["0"]["recall"] >= 0.93,
    "reduced_precision": metric_report["1"]["precision"] >= 0.88,
    "reduced_recall": metric_report["1"]["recall"] >= 0.90,
    "congested_precision": metric_report["2"]["precision"] >= 0.90,
    "congested_recall": metric_report["2"]["recall"] >= 0.85,
    "direct_normal_congested_error": direct_extreme_error <= 0.01,
    "ece": ece <= 0.05,
}
for gate_name, passed in metric_gates.items():
    if not passed:
        promotion_blockers.append(f"metric gate failed: {gate_name}")
congested_test = test_frame.loc[test_frame["traffic_state"].eq(2)]
congested_clips = build_group_ids(congested_test).nunique() if not congested_test.empty else 0
if len(congested_test) < 100 or congested_clips < 20:
    promotion_blockers.append(f"Congested support insufficient: {len(congested_test)} minutes / {congested_clips} clips")
if negative_exposure_hours < 300:
    promotion_blockers.append(f"incident negative exposure insufficient: {negative_exposure_hours:.2f}/300 h")
elif false_candidates_per_hour >= 0.01:
    promotion_blockers.append("incident candidate rate is not below 1 per 100 hours")
production_eligible = not promotion_blockers and TRAINING_MODE is TrainingMode.HITL_RETRAINING
training_lifecycle = build_training_lifecycle(TRAINING_MODE, MODEL_INPUT_POLICY, production_eligible=production_eligible)


### 9.1 Sellar procedencia y escribir el manifiesto

Esta celda crea el comprobante exacto de los datos usados (`training input lock`), escribe el manifiesto y muestra por qué el modelo quedó como piloto, candidato o producción.

In [ ]:
# Cell 9b — Input lock, manifest, and export summary
training_input_lock = create_training_input_lock(
    TRAINING_RUNS_ROOT,
    training_pipeline_run_id=TRAINING_PIPELINE_RUN_ID,
    training_mode=TRAINING_MODE.value,
    seed_snapshot=seed_snapshot.descriptor if seed_snapshot is not None else None,
    hitl_catalog=HITL_CATALOG_DESCRIPTOR,
    human_holdout=human_holdout_snapshot.descriptor if human_holdout_snapshot is not None else None,
    result_rows={"train": len(train_frame), "validation": len(validation_frame), "test": len(test_frame)},
    resolution={"validated_feedback": len(validated_feedback), "confirmed_incidents": len(confirmed_incidents), "discarded_proxy_rows": discarded_proxy_rows, "catalog_duplicate_rows": sum(HITL_CATALOG_DESCRIPTOR.get("duplicate_rows_resolved", {}).values()) if HITL_CATALOG_DESCRIPTOR else 0, "catalog_corrections": HITL_CATALOG_DESCRIPTOR.get("corrections_resolved", 0) if HITL_CATALOG_DESCRIPTOR else 0},
)
print(f"🔐 Training input lock → {training_input_lock.path}")
create_manifest(
    _MODEL_DIR,
    metrics={"f1_macro": float(f1_macro), "weak_supervision_fidelity": weak_supervision_fidelity, "human_holdout_direct_accuracy": human_holdout_direct_accuracy, "expected_confusion_cost": confusion_cost, "ece": ece, "brier_score": brier, "direct_normal_congested_error": direct_extreme_error, "automatic_accident_states": automatic_accident_states, "incident_candidate_count": reliable_incident_candidates, "negative_exposure_hours": negative_exposure_hours, "false_candidates_per_hour": false_candidates_per_hour, "synthetic_incident_episode_sensitivity": synthetic_incident_episode_sensitivity, "confirmed_incident_support": confirmed_incident_support, "confirmed_incident_candidate_sensitivity": confirmed_incident_candidate_sensitivity, "selected_balance_strategy": SELECTED_BALANCE_STRATEGY.value, "balance_validation_candidates": balance_selection_report.to_dict(orient="records"), "production_eligible": production_eligible},
    data_provenance={
        "origin": "training-notebook",
        "dataset_source": str(DATA_SOURCE),
        "record_count": int(len(df_features)),
        "real_record_count_before_engineering": int(_n_before),
        "synthetic_record_count_before_engineering": int(_n_synthetic),
        "synthetic_data_included": bool(synthetic_train.any()),
        "synthetic_records_in_validation": 0,
        "synthetic_records_in_test": 0,
        "telemetry_v2_coverage": dataset_audit.report["telemetry_v2_coverage"],
        "human_test_only": human_test_only,
        "human_holdout": human_holdout,
        "production_eligible": production_eligible,
        "promotion_blockers": promotion_blockers,
    },
    decision_policy=decision_policy,
    training_lifecycle=training_lifecycle,
    human_holdout=human_holdout_snapshot.descriptor if human_holdout_snapshot is not None else None,
    training_input_lock=training_input_lock.descriptor,
)
print(f"\nEtapa del modelo: {training_lifecycle['deployment_stage'].upper()}")
for blocker in promotion_blockers:
    print(f"   - {blocker}")

print(f"\n💾 Artefactos exportados:")
print(f"   Modelo   → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Etiquetas → {os.path.abspath(label_path)} (clases: {list(label_mapping.values())})")
print(f"   Scaler → {os.path.abspath(os.path.join(_MODEL_DIR, 'feature_scaler.joblib'))}")

print("\n📝 Notas sobre el soporte:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")

## 10. Validación cruzada opcional

Entrena cinco veces con grupos diferentes para comprobar si el resultado es estable o depende demasiado de una única partición.

- **StratifiedGroupKFold**: mantiene clips completos dentro de cada fold
- El scaler y los class weights se ajustan exclusivamente en el train de cada fold
- Reporta media y desviación de F1-macro y marca folds sin soporte suficiente

In [ ]:
# Cell 7b — K-Fold Cross-Validation (Optional)
#
# Run this cell for a more robust generalization estimate.
# It does NOT replace the single-split model (exported in Cell 7).

from sklearn.model_selection import StratifiedGroupKFold

cv_source = validated_feedback if TRAINING_MODE is TrainingMode.HITL_RETRAINING else df_features
cv_frame = cv_source.loc[~cv_source.get("data_origin", pd.Series("real", index=cv_source.index)).eq("synthetic")].copy()
groups_all = build_group_ids(cv_frame).values
N_FOLDS: int = min(5, len(np.unique(groups_all)))
if N_FOLDS < 2:
    raise RuntimeError("La validación cruzada agrupada necesita al menos dos grupos reales.")
kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Only real, unscaled records can enter validation folds.
X_all = apply_model_input_policy(cv_frame, MODEL_INPUT_POLICY).values
y_all = cv_frame["traffic_state"].values

fold_f1_scores: list[float] = []

print(f"🔁 Validación cruzada estratificada en {N_FOLDS} folds")
print("=" * 50)


In [ ]:

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all, y_all, groups_all), 1):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    # Scale per fold
    fold_scaler = StandardScaler()
    X_tr = fold_scaler.fit_transform(X_tr)
    X_val = fold_scaler.transform(X_val)

    fold_classes = np.unique(y_tr)
    fold_balanced = compute_class_weight(class_weight="balanced", classes=fold_classes, y=y_tr)
    fold_weights = {int(code): min(float(weight), 4.0) for code, weight in zip(fold_classes, fold_balanced)}
    fold_sample_weight = np.array([fold_weights[int(code)] for code in y_tr])

    # Build the exact same canonical architecture used by the final candidate.
    fold_model = build_traffic_state_mlp(
        input_features=X_tr.shape[1], output_classes=N_MODEL_STATES
    )
    fold_model.fit(
        X_tr, y_tr,
        epochs=200,
        batch_size=32,
        validation_data=(X_val, y_val),
        sample_weight=fold_sample_weight,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
        ],
        verbose=0,
    )

    y_pred_fold = fold_model.predict(X_val, verbose=0).argmax(axis=1).astype(int)

    fold_support = {code: int((y_val == code).sum()) for code in range(N_MODEL_STATES)}
    missing_fold_classes = [STATE_LABELS[code] for code, support in fold_support.items() if support == 0]
    fold_f1 = f1_score(y_val, y_pred_fold, labels=[0, 1, 2], average="macro", zero_division=0)
    fold_f1_scores.append(np.nan if missing_fold_classes else fold_f1)
    status = f"INSUFICIENTE faltan={missing_fold_classes}" if missing_fold_classes else f"F1-macro={fold_f1:.4f}"
    print(f"   Fold {fold_idx}: {status} | soporte={fold_support}")

evaluable_fold_scores = np.asarray(fold_f1_scores, dtype=float)
mean_f1 = np.nanmean(evaluable_fold_scores) if np.isfinite(evaluable_fold_scores).any() else float("nan")
std_f1 = np.nanstd(evaluable_fold_scores) if np.isfinite(evaluable_fold_scores).any() else float("nan")

print("=" * 50)
print(f"📊 K-Fold F1-macro: {mean_f1:.4f} ± {std_f1:.4f}")


In [ ]:

if mean_f1 >= 0.88:
    print(f"✅ El F1-macro cruzado cumple el objetivo (≥ 0.88)")
else:
    print(f"⚠️ El F1-macro cruzado no alcanza el objetivo (≥ 0.88)")
    print(f"   Revisá calidad de etiquetas y soporte por clase antes de cambiar el modelo.")

## 11. Copiar el bundle a Google Drive

En Colab, esta etapa copia los cuatro archivos terminados a Drive para que inferencia pueda cargarlos después de reiniciar el runtime. En local se omite sin modificar artefactos.

In [ ]:
# Cell 7c — Export Artifacts to Google Drive (Colab only)
#
# Copies trained artifacts to Google Drive so the inference workflow can load them
# even after a Colab runtime reset.  Skipped silently when running locally.

import shutil

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)

        _drive_dest = os.path.join("/content/drive", DRIVE_ARTIFACT_DIR)
        os.makedirs(_drive_dest, exist_ok=True)

        _artifact_files = [
            os.path.join(_MODEL_DIR, "traffic_classifier.keras"),
            os.path.join(_MODEL_DIR, "feature_scaler.joblib"),
            os.path.join(_MODEL_DIR, "label_mapping.joblib"),
            os.path.join(_MODEL_DIR, MANIFEST_FILE),
        ]

        _copied = 0
        for src_path in _artifact_files:
            if os.path.isfile(src_path):
                shutil.copy2(src_path, _drive_dest)
                _copied += 1
            else:
                print(f"⚠️ No encontrado; se omite: {src_path}")

        if _copied == len(_artifact_files):
            print(f"✅ {_copied} artefactos copiados a Google Drive:")
            print(f"   {_drive_dest}")
        else:
            print(f"⚠️ Sólo se copiaron {_copied}/{len(_artifact_files)} artefactos")
    except Exception as e:
        print(f"⚠️ No se pudo copiar a Drive: {e}")
        print("   Los artefactos siguen disponibles localmente para inferir en esta sesión.")
else:
    print("ℹ️ Entorno local: se omite la copia a Drive.")
    print(f"   Artefactos en: {os.path.abspath(_MODEL_DIR)}")

## 12. Límite de persistencia operacional

El perfil `training` es estrictamente read-only. Este notebook genera artefactos locales/Drive/DVC y no escribe predicciones operacionales. La persistencia pertenece exclusivamente al workflow de inferencia.

La base operativa se organiza en `vaaet_raw.traffic_data`, `vaaet_ml.telemetry_features`, `vaaet_ml.traffic_predictions` y `vaaet_feedback.human_validations`. Las validaciones humanas son append-only y nunca son modificadas por un reentrenamiento.

In [ ]:
# Cell 8 — Read-only training boundary
print("✅ Entrenamiento terminado sin escrituras en la base operacional.")
print(f"   Artefactos: {os.path.abspath(_MODEL_DIR)}")
print("   Siguiente paso: usá analyze_traffic_video.ipynb para probar el bundle y generar feedback.")
